In [1]:
import os, json, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
import joblib

# Cargar configuración
with open("../config.json") as f:
    config = json.load(f)

df = pd.read_csv(os.path.join("..", config["data"]["path"]))
target = config["data"]["target_column"]

# Feature engineering
df["Vehicle_Age"] = 2025 - df["Year"]
df.drop(columns=["Year", "Car_Name"], inplace=True)

X = df.drop(columns=[target])
y = df[target]

# Identificar columnas
categorical = X.select_dtypes(include="object").columns.tolist()
numerical = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical),
    ("cat", cat_pipeline, categorical)
])

# Modelo completo
final_model = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", GradientBoostingRegressor())
])

# Entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
final_model.fit(X_train, y_train)

# Guardar el modelo
os.makedirs("models", exist_ok=True)
joblib.dump(final_model, "models/best_model.pkl")

# Guardar histórico para monitoreo
os.makedirs("data", exist_ok=True)
X_train.to_csv("data/historico.csv", index=False)

